In [1]:
import pandas as pd
import plotly.express as px

In [2]:
url = "https://raw.githubusercontent.com/HecVelaz/Proyecto-Mburica-/main/03_Datos_obtenidos/Mburicao/Mburicao_Sil.csv"

df0 = pd.read_csv(url, header=None, names=["fecha","nivel"])

In [3]:
inicio = '2025-10-01 00:00:00'
fin = '2025-10-31 23:55:00'
mascara = (df0['fecha'] >= inicio) & (df0['fecha'] <= fin)
df_octubre = df0.loc[mascara].copy()

In [4]:
df_octubre['fecha'] = pd.to_datetime(df_octubre['fecha'])
df_octubre= df_octubre.sort_values('fecha').reset_index(drop=True)

In [5]:
df_octubre

,fecha,nivel
0,2025-10-01 00:00:00,6.504
1,2025-10-01 00:05:00,6.505
2,2025-10-01 00:10:00,6.503
3,2025-10-01 00:15:00,6.506
4,2025-10-01 00:20:00,6.505
...,...,...
8787,2025-10-31 23:35:00,6.551
8788,2025-10-31 23:40:00,6.548
8789,2025-10-31 23:45:00,6.547
8790,2025-10-31 23:50:00,6.550


In [6]:
# Contar ocurrencias de cada fecha
conteo = df_octubre['fecha'].value_counts()

# Filtrar aquellas que aparecen más de una vez
repetidas = conteo[conteo > 1]

print(f"Total de fechas con repeticiones: {len(repetidas)}")
print("\nFechas repetidas y su frecuencia:")
print(repetidas)

# Opcional: mostrar las filas correspondientes a esas fechas para inspeccionar
if not repetidas.empty:
    # Crear una lista de las fechas repetidas
    fechas_repetidas = repetidas.index.tolist()
    # Filtrar el DataFrame original por esas fechas y ordenar
    df_repetidas = df_octubre[df_octubre['fecha'].isin(fechas_repetidas)].sort_values('fecha')
    display(df_repetidas)

Total de fechas con repeticiones: 45

Fechas repetidas y su frecuencia:
fecha
2025-10-29 04:00:00    2
2025-10-23 04:30:00    2
2025-10-20 04:55:00    2
2025-10-12 11:55:00    2
2025-10-22 12:55:00    2
2025-10-07 13:20:00    2
2025-10-19 03:10:00    2
2025-10-05 04:35:00    2
2025-10-30 02:50:00    2
2025-10-02 03:25:00    2
2025-10-22 14:05:00    2
2025-10-08 03:10:00    2
2025-10-23 08:45:00    2
2025-10-23 11:35:00    2
2025-10-25 02:45:00    2
2025-10-03 03:30:00    2
2025-10-29 09:50:00    2
2025-10-21 07:15:00    2
2025-10-24 02:10:00    2
2025-10-03 08:20:00    2
2025-10-22 03:35:00    2
2025-10-11 07:40:00    2
2025-10-01 05:05:00    2
2025-10-22 16:55:00    2
2025-10-31 03:45:00    2
2025-10-26 02:20:00    2
2025-10-13 04:50:00    2
2025-10-27 02:45:00    2
2025-10-22 20:55:00    2
2025-10-18 02:05:00    2
2025-10-09 07:50:00    2
2025-10-04 03:30:00    2
2025-10-18 20:20:00    2
2025-10-15 03:05:00    2
2025-10-08 10:40:00    2
2025-10-16 02:25:00    2
2025-10-28 02:30:00   

,fecha,nivel
59,2025-10-01 05:05:00,6.530
60,2025-10-01 05:05:00,6.530
324,2025-10-02 03:25:00,6.543
325,2025-10-02 03:25:00,6.543
613,2025-10-03 03:30:00,6.548
...,...,...
8081,2025-10-29 11:55:00,6.514
8257,2025-10-30 02:50:00,6.551
8258,2025-10-30 02:50:00,6.551
8550,2025-10-31 03:45:00,6.560


In [7]:
# =============================================
# ELIMINAR FILAS CON FECHA DUPLICADA (conservar la primera)
# =============================================

# Verificar cuántas filas tenemos antes
print(f"Filas antes de eliminar duplicados: {len(df_octubre)}")

# Identificar fechas repetidas (opcional, solo para diagnóstico)
duplicados_fecha = df_octubre.duplicated(subset=['fecha'], keep=False)
print(f"Filas con fecha duplicada (incluyendo primera aparición): {duplicados_fecha.sum()}")
if duplicados_fecha.sum() > 0:
    print("Ejemplo de fechas repetidas (primeras 5):")
    display(df_octubre[duplicados_fecha].sort_values('fecha').head(10))

# Eliminar duplicados basados en la columna 'fecha', conservando la primera ocurrencia
df_octubre = df_octubre.drop_duplicates(subset=['fecha'], keep='first')

# Verificar después
print(f"Filas después de eliminar duplicados: {len(df_octubre)}")

Filas antes de eliminar duplicados: 8792
Filas con fecha duplicada (incluyendo primera aparición): 90
Ejemplo de fechas repetidas (primeras 5):


,fecha,nivel
59,2025-10-01 05:05:00,6.530
60,2025-10-01 05:05:00,6.530
324,2025-10-02 03:25:00,6.543
325,2025-10-02 03:25:00,6.543
613,2025-10-03 03:30:00,6.548
614,2025-10-03 03:30:00,6.548
672,2025-10-03 08:20:00,6.550
673,2025-10-03 08:20:00,6.550
900,2025-10-04 03:30:00,6.550
901,2025-10-04 03:30:00,6.550


Filas después de eliminar duplicados: 8747


In [8]:
# Contar cuántos duplicados quedan (debe ser 0)
print("Número de filas con fecha duplicada:", df_octubre.duplicated(subset=['fecha']).sum())

Número de filas con fecha duplicada: 0


In [9]:
# =============================================
# Crear índice regular de 5 minutos y reindexar
# =============================================

# Asegurar que la columna 'fecha' sea el índice
df_octubre = df_octubre.set_index('fecha').sort_index()

# Definir rango completo
start_time = df_octubre.index.min()
end_time = df_octubre.index.max()
time_index = pd.date_range(start=start_time, end=end_time, freq='5min')

# Reindexar: las marcas faltantes quedarán con NaN
df_regular = df_octubre.reindex(time_index)
df_regular.index.name = 'fecha'

# Verificar cuántos NaN se han introducido
print(f"Filas después de reindexar: {len(df_regular)}")
print(f"Valores nulos en 'nivel': {df_regular['nivel'].isna().sum()}")

Filas después de reindexar: 8928
Valores nulos en 'nivel': 181


In [10]:
# Interpolar linealmente los valores nulos en la columna 'nivel'
df_regular['nivel'] = df_regular['nivel'].interpolate(method='linear')

# Verificar que ya no hay nulos
print("Valores nulos después de interpolar:", df_regular['nivel'].isna().sum())

Valores nulos después de interpolar: 0


In [11]:
# =============================================
#  APLICAR  (7 - nivel)
#
# =============================================
df_regular['nivel'] = 7 - df_regular['nivel']

In [12]:
# =============================================
# VERIFICAR FRECUENCIA UNIFORME (5 minutos)
# =============================================
diffs = df_regular.index.to_series().diff()
print("\nFrecuencia de las diferencias entre timestamps (debe ser 5 minutos):")
print(diffs.value_counts())


Frecuencia de las diferencias entre timestamps (debe ser 5 minutos):
fecha
0 days 00:05:00    8927
Name: count, dtype: int64


In [13]:
# =============================================
# GRAFICAR INTERACTIVAMENTE CON PLOTLY
# =============================================
import plotly.express as px

fig = px.line(df_regular, x=df_regular.index, y='nivel',
              title='Nivel de agua (m) – Octubre 2025 ')
fig.update_layout(hovermode='x unified')
fig.show()

In [14]:
import pandas as pd
from google.colab import files

df_regular['nivel'] = df_regular['nivel'].round(4)

archivo_excel = '/content/Mburicao_Octubre_5min.xlsx'
archivo_csv   = '/content/Mburicao_Octubre_5min.csv'

df_regular.to_excel(archivo_excel, index=True)
df_regular.to_csv(archivo_csv, index=True)

files.download(archivo_excel)
files.download(archivo_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>